← [Overview](00_overview.ipynb)

# Partitional clustering: k-means, k-medoids, k-maxoids

Partitional clustering assigns every period to **exactly one** cluster and describes each
cluster by a single representative period. All three methods below share the same two
ingredients — the **distance** of section 2 and the rule *"assign every period to its
nearest representative"* — and differ only in **how they choose the representatives**:

| Method | Representatives chosen to… | How it is solved |
|---|---|---|
| **k-means** | minimise the total within-cluster distance (*approximately*) | Lloyd's heuristic → local optimum |
| **k-medoids** | minimise that same distance, centres restricted to real periods (*exactly*) | MILP → global optimum (needs a solver) |
| **k-maxoids** | **maximise** the distance *between* the centres (spread) | greedy heuristic |

So k-means and k-medoids chase the *same* objective — one only approximately, the other to
proven optimality — while k-maxoids deliberately does the opposite, picking maximally
different periods to keep diversity at the cost of accuracy.

> **Clustering vs. representation are two separate steps.** Choosing the *partition* (which
> periods group together) and choosing each cluster's *representative profile* (the mean, a
> real day, a re-sorted duration curve, …) are independent in tsam. The three methods here
> only set a sensible **default** representative; you can override it freely — see
> [Representation](06_representation.ipynb).

Partitional clustering sits in the **feature-based / typical-periods** quadrant of the
Hoffmann (2020) taxonomy.

---

## 1  The data: six periods as points

The [preprocessing notebook](01_preprocessing.ipynb) already did the work every clustering
method depends on: it normalised each attribute to $[0, 1]$ and **unstacked** the flat
series so each of the six days becomes one row-vector.

Each row is one period: a single point in the $N_a \cdot N_t = 2 \times 4 = 8$-dimensional
feature space. Clustering groups these six points. The whole notebook stays on this tiny
set, so every number can be checked by hand.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam
from tsam import ClusterConfig

pio.renderers.default = "notebook_connected"

# Preprocessed period matrix D (normalised + unstacked) from 01_preprocessing.
D = pd.read_csv("../tiny_periods.csv", header=[0, 1], index_col=0)
D_arr = D.values                       # shape (6, 8): six periods, eight features
N_PERIODS = D_arr.shape[0]
N_ATTRS, N_TIMESTEPS = 2, 4

# Raw tiny series — for the tsam.aggregate calls (it normalises internally).
tiny = pd.read_csv("../tiny.csv", index_col=0, parse_dates=True)

print("period matrix D:", D_arr.shape)
D.round(4)

period matrix D: (6, 8)


solar                       load                        
TimeStep      0      1      2    3       0       1       2       3
PeriodNum                                                         
0           0.0  1.000  0.750  0.0  0.0000  0.0000  0.1429  0.2857
1           0.0  0.875  0.875  0.0  0.0000  0.0000  0.1429  0.1429
2           0.0  0.375  0.250  0.0  0.1429  0.2857  0.2857  0.4286
3           0.0  0.250  0.375  0.0  0.1429  0.1429  0.4286  0.2857
4           0.0  0.125  0.125  0.0  0.4286  0.4286  0.5714  0.5714
5           0.0  0.125  0.000  0.0  0.4286  0.5714  0.7143  1.0000

---

## 2  The objective: the distance that gets minimised in k-means and k-medoids

The shared building block of k-means and k-medoids is the **distance between a period and a
centre**. Writing $x_{p,a,t}$ for the value of attribute $a$ at timestep $t$ in period $p$:

$$
\text{dist}(x_p, c_k) = \sqrt{\sum_{a=1}^{N_a} \sum_{t=1}^{N_t} (x_{p,a,t} - c_{k,a,t})^2}
$$

The double sum walks **every coordinate** of the period vector:

* $a = 1 \dots N_a$ indexes the **attributes** — here $N_a = 2$ (`solar`, `load`);
* $t = 1 \dots N_t$ indexes the **timesteps within a period** — here $N_t = 4$ (the four
  6-hourly steps of a day).

so it runs over all $N_a \cdot N_t = 2 \times 4 = 8$ coordinates — exactly the eight columns
of $D$.

This is one **element-wise** subtraction of the two 8-vectors: coordinate $(a,t)$ of the
period is only ever compared with the **same** $(a,t)$ of the centre — `solar@t1` against
`solar@t1`, never a different timestep or attribute. You do *not* feed in whole periods and
get cross-terms; you subtract matching coordinates, square the eight gaps, sum, and take the
root. So $x_p$ and $c_k$ are points of the same shape, and a centre is directly comparable to
a period whether it is a synthetic mean or a real day.

From this single distance, the quality of a whole clustering is the **total within-cluster
distance** $J$ — every period summed against the centre of the cluster it lands in:

$$
J = \sum_{k=1}^{N_k} \sum_{p \in \mathbb{C}_k} \text{dist}(x_p, c_k)^2
$$

| Symbol | Meaning |
|---|---|
| $x_p$ | period $p$ — one row of $D$ (`D_arr[p]`), a point in 8-D space |
| $\mathbb{C}_k$ | **cluster $k$**: the set of periods assigned to group $k$ |
| $c_k$ | the **centre** of cluster $k$ |
| $N_k$ | number of clusters ( = `n_clusters`) |

$J$ is the yardstick k-means and k-medoids both drive down; section 5 shows how k-maxoids
treats the same distance differently.

In [2]:
def euclidean_dist(x_p, c_k):
    """Euclidean distance between a period and a cluster center.

    x_p, c_k : np.ndarray, shape (N_a * N_t,) == (8,)
        A period and a center — same shape and coordinate order, one entry per
        (attribute a, timestep t) pair.

    Returns the scalar sqrt(sum((x_p - c_k) ** 2)); zero only for identical vectors.
    """
    return float(np.sqrt(np.sum((x_p - c_k) ** 2)))


# A concrete pair: period 0 vs a stand-in centre (period 2).
x_p = D_arr[0]   # day 0 (a sunny day)
c_k = D_arr[2]   # day 2 (an overcast day), used here as a stand-in centre

# Label every coordinate with the (a, t) it belongs to — the 8 columns of D.
terms = pd.DataFrame({
    "a (attribute)": D.columns.get_level_values(0),
    "t (timestep)": D.columns.get_level_values(1).astype(int),
    "x_p": x_p,
    "c_k": c_k,
    "gap = x_p - c_k": x_p - c_k,
    "gap**2": (x_p - c_k) ** 2,
})
print(terms.round(3).to_string(index=False))

# One term of the double sum picked out, e.g. a = solar, t = 1 (coordinate index 1):
print("\nExample single term  a=solar, t=1:",
      f"(x = {x_p[1]:.3f} - c = {c_k[1]:.3f})**2 = {(x_p[1] - c_k[1])**2:.3f}")
print("sum over all", N_ATTRS * N_TIMESTEPS, "terms      =",
      round(float(((x_p - c_k) ** 2).sum()), 4))
print("dist(x_p, c_k) = sqrt(sum)        =", round(euclidean_dist(x_p, c_k), 4))

a (attribute)  t (timestep)   x_p   c_k  gap = x_p - c_k  gap**2
        solar             0 0.000 0.000            0.000   0.000
        solar             1 1.000 0.375            0.625   0.391
        solar             2 0.750 0.250            0.500   0.250
        solar             3 0.000 0.000            0.000   0.000
         load             0 0.000 0.143           -0.143   0.020
         load             1 0.000 0.286           -0.286   0.082
         load             2 0.143 0.286           -0.143   0.020
         load             3 0.286 0.429           -0.143   0.020

Example single term  a=solar, t=1: (x = 1.000 - c = 0.375)**2 = 0.391
sum over all 8 terms      = 0.7835
dist(x_p, c_k) = sqrt(sum)        = 0.8851


---

## 3  Approximate solution — k-means (Lloyd's algorithm)

**Mechanism:** Lloyd's algorithm does not solve the objective exactly; it converges to a
*local* optimum by alternating two cheap steps until assignments stop changing:

1. Initialise $k$ centres $c_1, \dots, c_k$ (e.g. k-means++).
2. **Assignment step:** put each period with its nearest centre,
   $\text{cluster}(p) = \arg\min_k \text{dist}(x_p, c_k)$.
3. **Update step:** move each centre to the mean of its members,
   $c_k = \frac{1}{|\mathbb{C}_k|} \sum_{p \in \mathbb{C}_k} x_p$.
4. Repeat from step 2.

Its default representative is therefore the cluster **mean** — a synthetic centroid that need
not match any real day.

**TSAM configuration for k-means:**

In [3]:
# K-means: feature-based clustering using Lloyd's algorithm.
# representation defaults to 'mean' (centroid) for kmeans.
cfg_kmeans = ClusterConfig(method="kmeans", representation="mean")
print(cfg_kmeans)

ClusterConfig(include_period_sums=False, method='kmeans', representation='mean', scale_by_column_means=False, solver='highs', use_duration_curves=False)


### From-scratch Lloyd iteration on the tiny series

With the distance function in hand, the assignment step is just "call `euclidean_dist` for
every period against every centre and take the nearest", and the update step is the
centroid mean. Tracing it on the six periods (deterministic start at days 0, 2, 4, $k=3$):

In [4]:
# Reuse euclidean_dist(x_p, c_k) defined above.
k = 3
# Deterministic initialisation: pick periods 0, 2, 4 as initial centers
centers = D_arr[[0, 2, 4]].copy().astype(float)

print("Initial centers (periods 0, 2, 4):")
for i, c in enumerate(centers):
    print(f"  c{i} = {c.round(3)}")

for iteration in range(6):
    assignments = np.array(
        [np.argmin([euclidean_dist(D_arr[p], centers[kk]) for kk in range(k)])
         for p in range(N_PERIODS)]
    )
    new_centers = np.array(
        [D_arr[assignments == kk].mean(axis=0) if (assignments == kk).any()
         else centers[kk]
         for kk in range(k)]
    )
    converged = np.allclose(centers, new_centers)
    centers = new_centers
    print(f"\nIteration {iteration + 1}: assignments = {assignments}")
    if converged:
        print("  Converged.")
        break

print("\nFinal cluster assignments:")
for p in range(N_PERIODS):
    print(f"  day_{p} -> cluster {assignments[p]}")

Initial centers (periods 0, 2, 4):
  c0 = [0.    1.    0.75  0.    0.    0.    0.143 0.286]
  c1 = [0.    0.375 0.25  0.    0.143 0.286 0.286 0.429]
  c2 = [0.    0.125 0.125 0.    0.429 0.429 0.571 0.571]

Iteration 1: assignments = [0 0 1 1 2 2]

Iteration 2: assignments = [0 0 1 1 2 2]
  Converged.

Final cluster assignments:
  day_0 -> cluster 0
  day_1 -> cluster 0
  day_2 -> cluster 1
  day_3 -> cluster 1
  day_4 -> cluster 2
  day_5 -> cluster 2


In [5]:
# Verify centroid formula: c_k = (1/|C_k|) * sum of members
cluster_0_members = D_arr[assignments == 0]
centroid_0 = cluster_0_members.mean(axis=0)

print("Cluster 0 members:")
for p in np.where(assignments == 0)[0]:
    d = euclidean_dist(D_arr[p], centroid_0)
    print(f"  day_{p}: dist to centroid = {d:.4f}")

manual_centroid = cluster_0_members.sum(axis=0) / len(cluster_0_members)
print(f"\nCentroid c_0 (mean of members): {centroid_0.round(4)}")
print(f"Manual sum/count check:         {manual_centroid.round(4)}")
print("Match:", np.allclose(centroid_0, manual_centroid))

Cluster 0 members:
  day_0: dist to centroid = 0.1136
  day_1: dist to centroid = 0.1136

Centroid c_0 (mean of members): [0.     0.9375 0.8125 0.     0.     0.     0.1429 0.2143]
Manual sum/count check:         [0.     0.9375 0.8125 0.     0.     0.     0.1429 0.2143]
Match: True


In [6]:
# tsam k-means on the tiny series (k=3) — reproduces the hand-traced partition.
result_km = tsam.aggregate(
    tiny, n_clusters=3, period_duration="1D", cluster=ClusterConfig(method="kmeans")
)
print("k-means assignments:", np.asarray(result_km.cluster_assignments),
      "  weighted RMSE:", round(result_km.accuracy.weighted_rmse, 4))

k-means assignments: [1 1 0 0 2 2]   weighted RMSE: 0.0633


C:\Users\j.belina\AppData\Local\miniforge3\envs\tsam_improve_reworked_notebooks\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


> **Note on centroids and extremes:** the k-means centroid is the *average* of its
> members, so an extreme day with very different neighbours pulls the centroid toward the
> middle — peaks are partially erased. k-medoids and extreme-period handling address this;
> see [Extreme periods](08_extreme_periods.ipynb).

---

## 4  Exact solution — k-medoids (MILP)

**Mechanism:** where Lloyd's algorithm only *approximates* the minimum of $J$, k-medoids
**solves it exactly**. Restricting every centre to be an **actual period** turns the search
into a finite combinatorial problem that can be written as a **Mixed-Integer Linear Program
(MILP)** and handed to a solver, which returns a *provably globally optimal* clustering
(here via HiGHS).

The formulation is the classic **$p$-median / facility-location** problem (known in spatial
planning as the *Hess model*): a binary variable $z_{i,j}=1$ assigns period $j$ to centre
$i$, the diagonal $z_{i,i}=1$ marks period $i$ as one of the $k$ open centres, and the
objective $\min \sum_{i,j} d_{i,j}\, z_{i,j}$ is exactly the total within-cluster distance
$J$. Because every representative is then a real observed period, the profile shapes are
always physically realistic.

**TSAM configuration for k-medoids:**

In [7]:
# K-medoids: each representative is an actual observed period (medoid).
# Uses MILP optimization — solver='highs' (default, open-source).
cfg_kmedoids = ClusterConfig(method="kmedoids", representation="medoid", solver="highs")
print(cfg_kmedoids)

ClusterConfig(include_period_sums=False, method='kmedoids', representation='medoid', scale_by_column_means=False, solver='highs', use_duration_curves=False)


In [8]:
# Tiny: compute medoids manually for the k=3 assignment found above
print("Manual medoid computation for k=3:")
for kk in range(k):
    member_idx = np.where(assignments == kk)[0]
    members = D_arr[member_idx]
    total_dists = [
        (sum(euclidean_dist(members[i], members[j]) for j in range(len(members))),
         member_idx[i])
        for i in range(len(members))
    ]
    medoid_period = min(total_dists, key=lambda x: x[0])[1]
    print(f"  cluster {kk}: members = day_{member_idx} -> medoid = day_{medoid_period}")

Manual medoid computation for k=3:
  cluster 0: members = day_[0 1] -> medoid = day_0
  cluster 1: members = day_[2 3] -> medoid = day_2
  cluster 2: members = day_[4 5] -> medoid = day_4


### Seeing the objective: feature space and the distance matrix

Both k-means and k-medoids minimise the *same* quantity — the total distance from each
period to its cluster center — and differ only in what the center may be. Two pictures make
that concrete on the tiny `k=3` trace.

The first plots the six days as points in **feature space** (the 8-D period vectors from
[preprocessing](01_preprocessing.ipynb), projected to 2-D). Every coloured line runs from a
day to its cluster center, and **its length is one term of the objective** — the algorithm
shrinks the total. The only difference between the panels: k-means places the center at the
*centroid* (★, a synthetic mean that floats between days), while k-medoids forces it onto a
*real day* (ringed).

In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Project the six 8-D day-vectors to 2-D with PCA (numpy SVD) ---
D_centered = D_arr - D_arr.mean(axis=0)
_u, _s, _vt = np.linalg.svd(D_centered, full_matrices=False)
components = _vt[:2]                       # first two principal axes
pts = D_centered @ components.T           # (6, 2) projected day points
explained = (_s**2 / (_s**2).sum())[:2].sum()

# k-means centroids (the converged `centers`) projected onto the same axes
centroids_2d = (centers - D_arr.mean(axis=0)) @ components.T

# k-medoids: the medoid of each cluster is a real day -> reuse its projected point
medoid_of_cluster = {}
for kk in range(k):
    members_idx = np.where(assignments == kk)[0]
    members = D_arr[members_idx]
    totals = [sum(euclidean_dist(members[i], members[j]) for j in range(len(members)))
              for i in range(len(members))]
    medoid_of_cluster[kk] = int(members_idx[int(np.argmin(totals))])

palette = ["#636EFA", "#EF553B", "#00CC96"]
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("k-means — center is the centroid (mean)",
                    "k-medoids — center is a real day (medoid)"),
)

for col, mode in [(1, "centroid"), (2, "medoid")]:
    # member -> center connectors: each line's length is one term of the objective
    for p in range(N_PERIODS):
        kk = assignments[p]
        cx, cy = centroids_2d[kk] if mode == "centroid" else pts[medoid_of_cluster[kk]]
        fig.add_trace(
            go.Scatter(x=[pts[p, 0], cx], y=[pts[p, 1], cy], mode="lines",
                       line=dict(color=palette[kk], width=1), opacity=0.5,
                       showlegend=False),
            row=1, col=col,
        )
    # the six days
    fig.add_trace(
        go.Scatter(
            x=pts[:, 0], y=pts[:, 1], mode="markers+text",
            text=[f"day_{p}" for p in range(N_PERIODS)], textposition="top center",
            marker=dict(size=11, color=[palette[assignments[p]] for p in range(N_PERIODS)]),
            showlegend=False,
        ),
        row=1, col=col,
    )
    # the centers
    if mode == "centroid":
        fig.add_trace(
            go.Scatter(x=centroids_2d[:, 0], y=centroids_2d[:, 1], mode="markers",
                       marker=dict(symbol="star", size=18, color=palette[:k],
                                   line=dict(color="black", width=1)),
                       showlegend=False),
            row=1, col=col,
        )
    else:
        med = [medoid_of_cluster[kk] for kk in range(k)]
        fig.add_trace(
            go.Scatter(x=pts[med, 0], y=pts[med, 1], mode="markers",
                       marker=dict(symbol="circle-open", size=22,
                                   line=dict(color="black", width=3)),
                       showlegend=False),
            row=1, col=col,
        )

fig.update_layout(
    title=f"The objective in feature space — each line is one distance term "
          f"(2-D PCA, {explained:.1%} of variance)",
    height=430, width=950,
)
fig.update_xaxes(title_text="PC 1")
fig.update_yaxes(title_text="PC 2", col=1)
fig.show()

The exact-MILP k-medoids in tsam never sees the day profiles at all. Its solver receives
**only the matrix of pairwise distances** between periods (`M.d` in the `_setup_k_medoids`
model in `k_medoids_exact.py`). Below is that full 6×6 matrix for the tiny dataset; the
outlined cell in each row is the distance that day contributes to the objective.

In [10]:
# The full 6x6 pairwise distance matrix — exactly what the k-medoids MILP receives as `M.d`.
Dmat = np.array(
    [[euclidean_dist(D_arr[i], D_arr[j]) for j in range(N_PERIODS)]
     for i in range(N_PERIODS)]
)
labels = [f"day_{p}" for p in range(N_PERIODS)]

fig = px.imshow(
    Dmat,
    x=labels,
    y=labels,
    text_auto=".2f",
    color_continuous_scale="Blues",
    labels=dict(x="candidate center", y="period", color="distance"),
    title="Pairwise distance matrix — the exact input to the k-medoids optimiser",
)

# Outline the cell each period contributes: day p -> the medoid of its cluster.
for p in range(N_PERIODS):
    m = medoid_of_cluster[assignments[p]]
    fig.add_shape(
        type="rect", x0=m - 0.5, x1=m + 0.5, y0=p - 0.5, y1=p + 0.5,
        line=dict(color="#EF553B", width=3),
    )
fig.update_layout(width=560, height=520)
fig.show()

objective = sum(Dmat[p, medoid_of_cluster[assignments[p]]] for p in range(N_PERIODS))
print(f"Objective = sum of outlined cells (day -> its medoid) = {objective:.3f}")

Objective = sum of outlined cells (day -> its medoid) = 1.021


The block structure is the whole story: days 0–1, 2–3 and 4–5 sit cheaply close (dark),
while crossing between blocks is expensive (light). The exact-MILP optimiser elects one
center per cluster and **assigns every period to it through a binary matrix $z_{i,j}$** —
the explicit form of the abstract "$p \in \mathbb{C}_k$" membership from section 2. It
minimises $\sum_{i,j} d_{i,j}\, z_{i,j}$, the sum of the outlined cells: here
$0.23 + 0.30 + 0.49 = 1.02$ (the diagonal *day → itself* terms are zero). The diagonal
entries it switches on ($z_{i,i}=1$) are exactly the days elected as medoids.

In [11]:
# tsam k-medoids on the tiny series (k=3).
result_kmed = tsam.aggregate(
    tiny, n_clusters=3, period_duration="1D", cluster=ClusterConfig(method="kmedoids")
)
print("k-medoids assignments:", np.asarray(result_kmed.cluster_assignments),
      "  weighted RMSE:", round(result_kmed.accuracy.weighted_rmse, 4))

k-medoids assignments: [0 0 1 1 2 2]   weighted RMSE: 0.0866


---

## 5  Maxoid selection — k-maxoids (partition only)

**Mechanism:** k-maxoids uses the **same distance** as the other two but turns the goal
around. For each of many random restarts it runs a greedy local search that repeatedly swaps
a period into the centre set whenever doing so **increases the total spread between the
centres**, $\sum_{i<j} \lVert m_i - m_j \rVert^2$ (each period may only replace the centre it
is nearest to). Across all restarts it then keeps the spread-maximised centre set with the
**lowest** $J$ — an inertia tie-break that, on well-separated data, often pulls it back onto
the very partition the J-minimisers find. Finally every period is assigned to its **nearest**
centre,

$$
\text{cluster}(p) = \arg\min_i \text{dist}(x_p, m_i),
$$

the same assignment rule as k-means and k-medoids. So the centres are real periods chosen for
**spread, not centrality**; where the result *does* differ, it trades reconstruction accuracy
for diversity.

(How the final *representative profile* is then built from each cluster — the "maxoid"
representation and its alternatives — is a separate choice, covered in
[Representation](06_representation.ipynb).)

**TSAM configuration for k-maxoids:**

In [12]:
# K-maxoids: selects the k most mutually dissimilar periods.
# representation defaults to 'maxoid' for kmaxoids.
cfg_kmaxoids = ClusterConfig(method="kmaxoids", representation="maxoid")
print(cfg_kmaxoids)

ClusterConfig(include_period_sums=False, method='kmaxoids', representation='maxoid', scale_by_column_means=False, solver='highs', use_duration_curves=False)


In [13]:
# tsam k-maxoids on the tiny series (k=3).
result_kmx = tsam.aggregate(
    tiny, n_clusters=3, period_duration="1D", cluster=ClusterConfig(method="kmaxoids")
)
print("k-maxoids assignments:", np.asarray(result_kmx.cluster_assignments),
      "  weighted RMSE:", round(result_kmx.accuracy.weighted_rmse, 4))
print("At k=3 the lowest-inertia tie-break makes k-maxoids agree on the natural pairs.")

k-maxoids assignments: [0 0 1 1 2 2]   weighted RMSE: 0.0838
At k=3 the lowest-inertia tie-break makes k-maxoids agree on the natural pairs.


---

## 6  Where the methods part ways: same six days, $k = 2$

At $k=3$ all three methods land on the natural pairs `{0,1} {2,3} {4,5}` — on
well-separated data, with a centre to spare for every group, the choices coincide. The
difference shows at **$k=2$**, where one centre must now serve more than one natural group:

* **k-means and k-medoids** minimise $J$, so they peel off the single most distinctive tight
  group — the **sunny pair** `{0,1}` — and lump the other four days together.
* **k-maxoids** anchors on the two **most extreme** days — the brightest (day 0) and the
  highest-load (day 5) — and assigns every day to the extreme it resembles, even splitting
  the two overcast middle days (2 and 3) to opposite clusters.

The spread-driven split reconstructs the series visibly worse — a higher $J$, hence the
larger RMSE in the table.

In [14]:
# All three at k=2 on the same six days (results are stable across seeds here).
def canon(labels):
    """Relabel clusters by first appearance so identical groupings look identical."""
    remap, out = {}, []
    for x in labels:
        remap.setdefault(x, len(remap))
        out.append(remap[x])
    return np.array(out)


np.random.seed(0)
rows, parts = [], {}
for method in ["kmeans", "kmedoids", "kmaxoids"]:
    r = tsam.aggregate(tiny, n_clusters=2, period_duration="1D",
                       cluster=ClusterConfig(method=method))
    parts[method] = canon(r.cluster_assignments)
    rows.append({"method": method,
                 "partition": str(parts[method]),
                 "weighted_rmse": round(r.accuracy.weighted_rmse, 4)})

print(pd.DataFrame(rows).to_string(index=False))
print("\nk-means and k-medoids agree: {0,1} | {2,3,4,5}")
print("k-maxoids differs:           {0,1,3} | {2,4,5}  (day 3 joins the bright cluster)")

  method     partition  weighted_rmse
  kmeans [0 0 1 1 1 1]         0.1223
kmedoids [0 0 1 1 1 1]         0.1318
kmaxoids [0 0 1 0 1 1]         0.1683

k-means and k-medoids agree: {0,1} | {2,3,4,5}
k-maxoids differs:           {0,1,3} | {2,4,5}  (day 3 joins the bright cluster)


In [15]:
# Show the k=2 partitions in the same 2-D feature space (reuses pts, explained).
pal2 = ["#636EFA", "#EF553B"]
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("k-means = k-medoids  (minimise J)", "k-maxoids  (maximise spread)"),
)
for col, method in [(1, "kmeans"), (2, "kmaxoids")]:
    lab = parts[method]
    fig.add_trace(
        go.Scatter(
            x=pts[:, 0], y=pts[:, 1], mode="markers+text",
            text=[f"day_{p}" for p in range(N_PERIODS)], textposition="top center",
            marker=dict(size=14, color=[pal2[int(l)] for l in lab],
                        line=dict(color="black", width=1)),
            showlegend=False,
        ),
        row=1, col=col,
    )
fig.update_layout(
    title=f"k = 2 partitions in feature space — colour = cluster "
          f"(2-D PCA, {explained:.0%} of variance)",
    height=430, width=900,
)
fig.update_xaxes(title_text="PC 1")
fig.update_yaxes(title_text="PC 2", col=1)
fig.show()

---

**Up next:**
* [Agglomerative clustering](03_agglomerative_clustering.ipynb) — hierarchical Ward and contiguous Ward
* [Representation](06_representation.ipynb) — choosing what the cluster center looks like
* [Extreme periods](08_extreme_periods.ipynb) — preserving peaks through clustering